In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
import numpy as np
from pathlib import Path

# Path.cwd() gets the current working directory (where your notebook lives)
# .parents[1] moves up two directories (equivalent to '../../')
project_root = Path.cwd().parents[1]

sys.path.insert(0, str(project_root.resolve()))
from src.simulations.GBM import (
    estimate_parameters,
    simulate_paths,
    simulate_paths2,
    simulate_correlated_GBM,
    estimate_Correlation_matrix
)
from src.data.statistics import compute_log_returns
# Assuming returns is your log-return dataframe
AAPL=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\AAPL.parquet")
NVDA=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\NVDA.parquet")
JPM=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\JPM.parquet")
SPY=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\SPY.parquet")
prices = pd.DataFrame({
    "AAPL": AAPL["adj_close"],
    "NVDA":NVDA['adj_close'],
    "JPM":JPM["adj_close"],
    "SPY":SPY['adj_close']
})
returns = compute_log_returns(prices)

In [2]:
prices = prices.dropna()
returns = compute_log_returns(prices).dropna()
prices.head()

,AAPL,NVDA,JPM,SPY
0,72.333893,5.963802,118.430305,296.888123
1,71.630638,5.868347,116.867470,294.640137
2,72.201401,5.892956,116.774536,295.764099
3,71.861847,5.964301,114.789291,294.932495
4,73.017830,5.975488,115.684784,296.504425


In [3]:
corr_matrix=returns.corr()
type(prices)

pandas.core.frame.DataFrame

In [4]:
n_assets=4
n_paths=5000
Z = np.random.normal(size=(n_assets, n_paths))
L = np.linalg.cholesky(corr_matrix)
correlated_Z = L @ Z
np.corrcoef(correlated_Z)

array([[1.        , 0.55495191, 0.40899341, 0.769432  ],
       [0.55495191, 1.        , 0.33424052, 0.69709763],
       [0.40899341, 0.33424052, 1.        , 0.70378707],
       [0.769432  , 0.69709763, 0.70378707, 1.        ]])

In [5]:
n_assets = 4
n_paths = 5000
n_steps = 252

S0 = prices.iloc[-1].to_numpy()

mu = (returns.mean() * 252).to_numpy()
sigma = (returns.std() * np.sqrt(252)).to_numpy()

paths = np.zeros((n_steps, n_assets, n_paths))

paths[0] = S0[:, None]

dt = 1/252

for t in range(1, n_steps):

    Z = np.random.normal(size=(n_assets, n_paths))
    Z_corr = L @ Z

    paths[t] = (
        paths[t-1]
        * np.exp(
            (mu[:, None] - 0.5 * sigma[:, None]**2) * dt
            + sigma[:, None] * np.sqrt(dt) * Z_corr
        )
    )

In [6]:
prices

,AAPL,NVDA,JPM,SPY
0,72.333893,5.963802,118.430305,296.888123
1,71.630638,5.868347,116.867470,294.640137
2,72.201401,5.892956,116.774536,295.764099
3,71.861847,5.964301,114.789291,294.932495
4,73.017830,5.975488,115.684784,296.504425
...,...,...,...,...
1602,298.970001,220.353180,295.700012,733.729980
1603,302.250000,223.209854,301.980011,741.250000
1604,304.989990,219.254456,303.000000,742.719971
1605,308.820007,215.079330,306.380005,745.640015


In [2]:
corraleted_matrix=estimate_Correlation_matrix(returns)
prices= pd.DataFrame({
    "AAPL": AAPL["adj_close"],
    "NVDA":NVDA['adj_close'],
    "JPM":JPM["adj_close"],
    "SPY":SPY['adj_close']
})
print(prices.head())
n_paths=5000
n_assets=4
years=3
steps_per_year=252
paths=simulate_correlated_GBM(
    corr_matrix=corraleted_matrix,
    prices=prices,
    returns=returns,
    n_paths=n_paths,
    n_assets=n_assets,
    years=years,
    steps_per_year=steps_per_year,

    )

        AAPL      NVDA         JPM         SPY
0  72.333893  5.963802  118.430305  296.888123
1  71.630638  5.868347  116.867470  294.640137
2  72.201401  5.892956  116.774536  295.764099
3  71.861847  5.964301  114.789291  294.932495
4  73.017830  5.975488  115.684784  296.504425
S0 and L okay
n_steps,mu,sigma okay


In [3]:
paths.shape

(756, 4, 5000)

In [ ]:
final_returns = np.log(paths[-1] / paths[-2])  # shape (assets, paths)

corr = np.corrcoef(final_returns)
print(corr)

[[1.         0.56944804 0.42547528 0.77981675]
 [0.56944804 1.         0.34932659 0.7118502 ]
 [0.42547528 0.34932659 1.         0.70672031]
 [0.77981675 0.7118502  0.70672031 1.        ]]
